# t-SNE from scratch avec NumPy : exercice

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/cercle-des-experts/blob/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2/formulation-implementation-algorithme-t-sne-partie-2-exercice.ipynb)

📖 **Le mini-cours complet** : [t-SNE depuis Zéro (Partie 2)](https://www.monshotdata.com/p/formulation-implementation-algorithme-t-sne-partie-2)

🧭 **Le cours** : [ML classique et Deep Learning › 2.3 Clustering et réduction de dimension](https://www.monshotdata.com/mini-cours/ml-classique-et-deep-learning#h2sc23)

---

Tu connais la théorie de t-SNE. Il est temps de la **coder toi-même**, sans scikit-learn, uniquement avec NumPy. Ce notebook contient le squelette complet : tu remplis les lignes marquées `À COMPLÉTER`, en suivant les étapes données en commentaires.

**À la fin de ce notebook, tu sauras :**

- calculer les probabilités conjointes en haute dimension, avec une variance adaptée à chaque point grâce à la perplexité ;
- implémenter la descente de gradient de t-SNE avec momentum et exagération précoce ;
- projeter dix clusters en 2D et vérifier visuellement que ton implémentation marche.


### Avant de lancer

**Comment faire l'exercice :** complète les lignes `À COMPLÉTER` dans les fonctions entre « TON CODE COMMENCE ICI » et « TON CODE SE TERMINE ICI », puis exécute le notebook de haut en bas. Si tu bloques, la solution est dans `formulation-implementation-algorithme-t-sne-partie-2-solution.ipynb` : essaie d'abord seul(e), c'est là qu'on apprend le plus.

Les données (`data.pickle`) et les images des formules sont téléchargées automatiquement. Pas de GPU nécessaire. Prévois quelques minutes pour l'entraînement.

🧪 *Testé le 20 novembre 2026 avec Python 3.13, NumPy, pandas et Matplotlib dans leurs versions de Colab.*

## 1. Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import warnings

warnings.filterwarnings("ignore")

### Les données

Le jeu de données fourni est une collection de points en haute dimension répartis en **dix clusters** (il a été créé ainsi). On connaît les étiquettes `y`, mais **t-SNE ne les utilise jamais** : elles ne servent qu'à colorier le résultat à la fin.

In [ ]:
# Télécharge les fichiers nécessaires depuis le dépôt GitHub s'ils ne sont pas déjà présents
import os, urllib.request

BASE = "https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2"
for f in ['data.pickle', 'images/image1.png', 'images/image2.png', 'images/image3.png', 'images/image4.png', 'images/image5.png']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{BASE}/{f}", f)

In [ ]:
df = pd.read_pickle("data.pickle")
X = df.drop(columns = ["y"]).values
y = df["y"]

In [ ]:
X.shape  ## Ensemble de données en haute dimension

In [ ]:
y.shape  ## Nous avons des étiquettes mais nous ne les utiliserons nulle part

On définit enfin quelques couleurs pour la visualisation finale.

In [ ]:
# Définir quelques couleurs pour la visualisation
colors = ['#fe7c73', '#2471A3', '#3498DB', '#27AE60', '#82E0AA', '#D35400', '#5D6D7E', '#E74C3C', '#21618C', '#B7950B', '#46C7C7', '#00B9FF', '#FF7051', "orange", "darkorange", "tomato", "coral", "limegreen", "lightsalmon"]

## 2. Étape 1 : les probabilités conjointes en haute dimension

Pour chaque point, t-SNE regarde ses voisins avec une gaussienne centrée sur lui, puis symétrise les résultats. La largeur de chaque gaussienne (sa variance σ²) est réglée **point par point** pour que la *perplexité* atteigne la valeur choisie : c'est ce qui permet de s'adapter aux zones denses comme aux zones clairsemées.

### Formule des probabilités conjointes en haute dimension

<img src="https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2/images/image1.png"  width="45%" height="25%">

### Formule de l'entropie et de la perplexité

<img src="https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2/images/image2.png"  width="35%" height="25%">

L'objectif est de définir une fonction ```find_high_dimensional_joint_probabilties``` qui accepte trois paramètres :

1. Un tableau numpy à n dimensions ```data```

2. L'hyperparamètre ```perplexity``` (perplexité)

3. ```variance_list``` pour trouver la valeur de variance optimale pour chaque point de données

Les étapes ont été mentionnées dans les commentaires

In [ ]:
def find_high_dimensional_joint_probabilties(data, perplexity, variance_list):
    """
    Calcule les probabilités conjointes p_ij en haute dimension pour t-SNE.
    
    Paramètres :
        data (numpy.ndarray) : Les points de données en haute dimension.
        perplexity (float) : L'hyperparamètre de perplexité.
        variance_list (list) : Liste des valeurs de variance à tester pour trouver le sigma^2 de chaque point.
    
    Retourne :
        numpy.ndarray : Les probabilités conjointes p_ij calculées en haute dimension.
    """
    
    # Définir le nombre de points de données
    num_data_points = data.shape[0]
    
    # Une matrice pour stocker les probabilités conjointes par paires
    high_dim_probabilities = np.zeros((num_data_points, num_data_points))
    
    # Itérer sur tous les points de données
    for i in range(num_data_points):
        # Sélectionner le point de données courant
        xi = data[i]
        
        # Déclarer des variables pour trouver sa meilleure valeur de variance
        best_variance = None
        min_perplexity_diff = float('inf')
        
        ## TON CODE COMMENCE ICI
        
        # Pour le point de données courant, calculer sa distance euclidienne à tous les autres points.
        distances = ...  # À COMPLÉTER
        
        ## TON CODE SE TERMINE ICI
        
        # Itérer sur toutes les variances dans la liste de variances
        for variance in variance_list:
            ## TON CODE COMMENCE ICI
            
            # Calculer les probabilités conjointes p_{ij} en haute dimension en utilisant
            # une densité de probabilité gaussienne avec variance = variance
            numerator = ...  # À COMPLÉTER
            
            # Mettre la distance à soi-même à zéro
            numerator[i] = ...  # À COMPLÉTER
            
            # Calculer le dénominateur de la densité de probabilité conditionnelle
            # en sommant toutes les densités individuelles
            denominator = ...  # À COMPLÉTER
            
            ## TON CODE SE TERMINE ICI
            
            # Trouver les densités conditionnelles finales pij
            pij = numerator / denominator
            
            ## TON CODE COMMENCE ICI
            
            # Calculer la perplexité de P(i)
            ## D'abord définir l'entropie
            # Note : Ajouter une petite valeur (10^-12) dans le log pour éviter les erreurs d'entrée invalide
            entropy = ...  # À COMPLÉTER
            
            ## Ensuite, trouver la perplexité du point de données donné
            perplexity_i = ...  # À COMPLÉTER
            
            ## TON CODE SE TERMINE ICI
            
            # Vérifier si la perplexité est la plus proche de la perplexité cible
            perplexity_diff = np.abs(perplexity_i - perplexity)
            
            if perplexity_diff < min_perplexity_diff:
                # Mettre à jour la best_variance avec la valeur de variance actuelle
                min_perplexity_diff = perplexity_diff
                best_variance = variance
        
        # Utiliser la best_variance pour calculer et stocker les probabilités en haute dimension pour xi
        high_dim_probabilities[i] = np.exp(-distances / (2.0 * best_variance))
    
    # Calculer maintenant les probabilités conjointes
    high_dim_probabilities = (high_dim_probabilities + high_dim_probabilities.T)/(2*num_data_points)
    
    # Mettre tous les pii à zéro
    np.fill_diagonal(high_dim_probabilities, 0)
    
    # Retourner les probabilités conjointes en haute dimension
    return high_dim_probabilities

## 3. Étape 2 : initialiser la projection en basse dimension

On place les points au hasard dans le plan (très proches de l'origine). t-SNE va ensuite les déplacer pas à pas.

In [ ]:
def inititalize_low_dimensional_mapping(num_data_points, n_components=2):
    """
    Initialise la projection y_i en basse dimension pour chaque point de données.
    
    Paramètres :
    - num_data_points (int) : Nombre de points de données (n).
    - n_components (int) : Nombre de dimensions réduites (d).
    
    Retourne :
    - low_dim_mapping (numpy.ndarray) : Projection initiale y_i en basse dimension pour chaque point de données.
    """
    
    ## TON CODE COMMENCE ICI
    
    # Créer un tableau numpy aléatoire de dimensions (num_data_points, n_components)
    low_dim_mapping = ...  # À COMPLÉTER
    
    ## TON CODE SE TERMINE ICI
    
    # Retourner la projection initiale en basse dimension
    return low_dim_mapping

## 4. Étape 3 : entraîner t-SNE

On compare maintenant les probabilités p_ij (haute dimension) à celles q_ij calculées dans le plan, et on déplace les points pour les rapprocher. Côté plan, on utilise une **loi de Student** à un degré de liberté : ses queues épaisses résolvent le problème d'encombrement (*crowding problem*).

### Formule des probabilités conjointes en basse dimension

<img src="https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2/images/image3.png"  width="45%" height="25%">

### Formule de la fonction de coût

Il s'agit de la divergence de Kullback-Leibler entre P et Q.

<img src="https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2/images/image4.png"  width="35%" height="25%">

### Formule des mises à jour des paramètres

Descente de gradient, avec un **momentum** et une **exagération précoce** (on multiplie temporairement les p_ij pour que les clusters se séparent nettement dès le début).

<img src="https://raw.githubusercontent.com/moncoachdata/cercle-des-experts/main/ml-classique-et-deep-learning/formulation-implementation-algorithme-t-sne-partie-2/images/image5.png"  width="65%" height="45%">

### La boucle d'entraînement

C'est la partie la plus longue de l'exercice : calcul des q_ij, du coût, du gradient et des mises à jour.

In [ ]:
def tsne_train(pij,
               low_dim_mapping,
               learning_rate=30,
               epochs=500,
               early_exaggeration_iterations=100,
               early_exaggeration_factor=4,
               lower_momentum_rate=0.5,
               upper_momentum_rate=0.8):
    """
    Entraîne l'algorithme t-SNE.
    
    Paramètres :
    - pij (numpy.ndarray) : Probabilités conjointes p_{ij} en haute dimension.
    - low_dim_mapping (numpy.ndarray) : Projection initiale y_i en basse dimension pour chaque point de données.
    - learning_rate (float) : Taux d'apprentissage pour la descente de gradient.
    - epochs (int) : Nombre total d'époques d'entraînement.
    - early_exaggeration_iterations (int) : Nombre d'époques pour appliquer l'exagération précoce.
    - early_exaggeration_factor (float) : Facteur d'exagération précoce à multiplier par p_{ij}.
    - lower_momentum_rate (float) : Taux de momentum pour les mises à jour des paramètres durant les premières étapes de l'entraînement.
    - upper_momentum_rate (float) : Taux de momentum pour les mises à jour des paramètres après les premières étapes de l'entraînement.
    
    Retourne :
    - low_dim_mapping (numpy.ndarray) : Projection finale y_i en basse dimension après entraînement.
    """
    
    # Définir le nombre de points de données
    num_data_points = low_dim_mapping.shape[0]
    
    # Variables pour stocker les deux valeurs précédentes des paramètres pour utiliser le momentum
    prev_low_dim_mapping = low_dim_mapping.copy()
    prev2_low_dim_mapping = low_dim_mapping.copy()
    
    # Itérer sur chaque époque
    for epoch in range(epochs):
        # Définir une matrice pour stocker les probabilités conjointes q_{ij} par paires en basse dimension
        qij = np.zeros((num_data_points, num_data_points))
        
        # Pour chaque point de données, calculer la distance euclidienne à tous les autres points
        for i in range(num_data_points):
            ## TON CODE COMMENCE ICI
            
            # Pour le point de données courant en basse dimension, calculer la distance euclidienne à tous les autres points
            distances = ...  # À COMPLÉTER
            
            # Trouver 1/(1+distances)
            inv_distances = ...  # À COMPLÉTER
            
            # Mettre la distance à soi-même à zéro
            inv_distances[i] = ...  # À COMPLÉTER
            
            ## TON CODE SE TERMINE ICI
            
            # Trouver q_ij[i] en divisant inv_distances par sa somme
            qij[i] = inv_distances / np.sum(inv_distances)
        
        # Calculer maintenant les probabilités conjointes
        qij = (qij + qij.T)/(2*num_data_points)
        
        # Mettre tous les qii à zéro
        np.fill_diagonal(qij, 0)
        
        ## TON CODE COMMENCE ICI
        
        # Appliquer l'exagération précoce si applicable
        if epoch < early_exaggeration_iterations:
            # Multiplier pij par le early_exaggeration_factor
            pij_new = ...  # À COMPLÉTER
            
            # Sélectionner le lower_momentum_rate
            momentum_rate = ...  # À COMPLÉTER
        
        else:
            # Garder pij tel quel
            pij_new = pij.copy()
            
            # Sélectionner le upper_momentum_rate
            momentum_rate = upper_momentum_rate  # Taux de momentum régulier
        
        # Calculer le coût
        # Note : Ajouter une petite valeur (10^-12) à pij et qij dans le log pour éviter les erreurs d'entrée invalide
        cost = ...  # À COMPLÉTER
        
        ## TON CODE SE TERMINE ICI
        
        # Tableau pour stocker le gradient pour chaque paramètre
        gradient = np.zeros_like(low_dim_mapping)
        
        # Pour chaque valeur de paramètre, trouver son gradient
        # Note : les paramètres sont précisément le tableau low_dim_mapping
        for i in range(num_data_points):
            ## TON CODE COMMENCE ICI
            
            # Trouver (yi-yj)
            diff = ...  # À COMPLÉTER
            
            # Trouver (pij-qij)
            qpi_diff = ...  # À COMPLÉTER
            
            # Trouver le dénominateur de l'équation du gradient
            d = ...  # À COMPLÉTER
            
            # Calculer le gradient
            grad_i = ...  # À COMPLÉTER
            
            # Le stocker pour les valeurs de paramètres actuelles
            gradient[i] = grad_i
            
            ## TON CODE SE TERMINE ICI
        
        ## TON CODE COMMENCE ICI
        
        # Mettre à jour la projection en basse dimension avec le momentum
        low_dim_mapping = ...  # À COMPLÉTER
        
        ## TON CODE SE TERMINE ICI
        
        # Mettre à jour les deux derniers paramètres pour le momentum
        prev2_low_dim_mapping = prev_low_dim_mapping.copy()
        prev_low_dim_mapping = low_dim_mapping.copy()
        
        # Afficher la fonction de coût
        if (epoch + 1) % 5 == 0:
            print(f"Époque {epoch + 1}/{epochs}, Coût : {cost}")
    
    return low_dim_mapping

### Calcul des probabilités conjointes en haute dimension

La perplexité fixe l'échelle du « voisinage » de chaque point (valeurs usuelles entre 5 et 50). Cette étape est la plus lente : une boucle sur tous les points et toutes les variances testées.

In [ ]:
pij = find_high_dimensional_joint_probabilties(X, perplexity=40, variance_list=np.linspace(0.1, 50, 100))

### Projection initiale

In [ ]:
low_dim_mapping = inititalize_low_dimensional_mapping(X.shape[0], 2)

### Entraînement et projection finale

On lance 500 époques, dont les 250 premières avec exagération précoce. Compte quelques dizaines de secondes à quelques minutes selon la machine : l'implémentation est volontairement en pur NumPy, sans optimisation.

In [ ]:
final_mapping = tsne_train(pij, low_dim_mapping, learning_rate=200, early_exaggeration_factor=4, 
                           early_exaggeration_iterations=250, epochs=500)

## 5. Visualiser le résultat

On colorie chaque point avec son étiquette d'origine. Si tout est correct, on doit voir dix groupes bien séparés.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(0.1*final_mapping[:, 0], 0.1*final_mapping[:, 1], c=[colors[i+3] for i in y])
ax.set_xlabel('Composante t-SNE 1', fontweight="bold", size=15)
ax.set_ylabel('Composante t-SNE 2', fontweight="bold", size=15)
ax.set_xticklabels([])
ax.set_yticklabels([])

## Ajuster en conséquence pour mieux visualiser !
ax.set_xlim(-1.5, 1.5) 
ax.set_ylim(-1.5, 1.5)
plt.show()

---

## Pour aller plus loin

- [t-SNE depuis Zéro (Partie 1)](https://www.monshotdata.com/p/formulation-implementation-algorithme-t-sne-partie-1) — toute la théorie, étape par étape
- [Construire l'algorithme de l'ACP de A à Z](https://www.monshotdata.com/p/algorithme-analyse-composantes-principales-acp-de-a-z) — l'autre grande technique de réduction de dimension
- [Gamma ou Gaussienne ? La réponse en une seule métrique.](https://www.monshotdata.com/p/divergence-kl) — la mesure que t-SNE minimise